# Segundo Examen Parcial - Control Difuso y Algoritmos Geneticos

 *Optimización Multiobjetivo de Sistemas Difusos mediante Algoritmos Geneticos con **PyGAD*** 

### **Objetivo**

Utilizar la biblioteca pyGAD para optimizar automáticamente sistemas de inferencia difusa tipo Mamdani empleados para el control de un servomotor de corriente directa.


#### Importaciones

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pygad
import skfuzzy as fuzz
from scipy.signal import lsim, lti, square
from skfuzzy import control as ctrl


#### Constantes obligatorias

In [2]:
# Universos de discurso
ERROR_MIN, ERROR_MAX = -np.pi/2, np.pi/2
DERROR_MIN, DERROR_MAX = -np.pi/6, np.pi/6
VOLTAGE_MIN, VOLTAGE_MAX = -8.18, 8.18
N_PUNTOS_UNIVERSO = 1000

# Simulación obligatoria
T_SIM = 200       # segundos
PERIODO = 100     # segundos, referencia cuadrada ±1 rad, ciclo útil 50 %

DT_REFERENCIA = 0.01              # paso fino contra el que se compara

### **Fase 1:** Implementación de los controladores base

- Controlador A. Sistema difuso con: **5 términos lingüísticos por entrada** (generados con automf)
- Controlador B. Sistema difuso con: **7 términos lingüísticos por entrada** (generados con automf)
- Controlador C. Sistema difuso con: **9 términos lingüísticos por entrada** (generados con automf) 

Cada variable tiene N términos generados con `automf` y la base usa todas las reglas (N² reglas).
`automf` solo trae nombres para 3, 5 y 7 términos, por eso se pasan los nombres explícitos.

#### Planta, tiempo y referencia

In [3]:
G = lti([0.5], [1, 13.6667, 23.3487, 0])

dt = DT_REFERENCIA
t = np.arange(0, T_SIM + dt, dt)

r = square(2*np.pi*t/PERIODO)

x0 = [0, 0, 0]

In [4]:
def crear_controlador(nombres):
    n = len(nombres)
    error = ctrl.Antecedent(np.linspace(ERROR_MIN, ERROR_MAX, 1000), 'error')
    derror = ctrl.Antecedent(np.linspace(DERROR_MIN, DERROR_MAX, 1000), 'derror')
    voltage = ctrl.Consequent(np.linspace(VOLTAGE_MIN, VOLTAGE_MAX, 1000), 'voltage')
    for variable in (error, derror, voltage):
        variable.automf(n, names=nombres)

    reglas = []
    for i in range(n):
        for j in range(n):
            k = i if i != n // 2 else j
            reglas.append(ctrl.Rule(error[nombres[i]] & derror[nombres[j]], voltage[nombres[k]]))
    return ctrl.ControlSystemSimulation(ctrl.ControlSystem(reglas))

controller_A = crear_controlador(['NB', 'NS', 'ZE', 'PS', 'PB'])
controller_B = crear_controlador(['NB', 'NM', 'NS', 'ZE', 'PS', 'PM', 'PB'])
controller_C = crear_controlador(['NVB', 'NB', 'NM', 'NS', 'ZE', 'PS', 'PM', 'PB', 'PVB'])

#### Simulación del lazo cerrado
En lazo cerrado el voltaje *$u$* depende de *$theta$*, así que no se puede llamar a `lsim` una sola vez con todo *$u$*.
Se llama a `lsim` en cada intervalo con el voltaje *$u_k$* constante y *$x_0$* igual al estado final del intervalo anterior. `lsim` toma *$x_0$* como el estado en $t = 0$, por eso cada intervalo se simula con `T = t[:2] = [0, dt]` y no con el tiempo absoluto del intervalo.

In [5]:
def paso_planta(u, t, x0):
    tout, theta, xout = lsim(G, U=u, T=t, X0=x0)
    return theta[-1], xout[-1]

def simular(controller):
    theta = np.zeros(len(t))
    u = np.zeros(len(t))
    x = x0
    for k in range(len(t) - 1):
        e = r[:k+1] - theta[:k+1]
        de = np.diff(e, prepend=e[0]) / dt
        controller.input['error'] = e[-1]
        controller.input['derror'] = de[-1]
        controller.compute()
        u[k] = controller.output['voltage']
        theta[k+1], x = paso_planta([u[k], u[k]], t[:2], x)
    return theta, u

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(t, r, 'k--', label='Referencia')
for nombre, controller in (('A (5 términos)', controller_A), ('B (7 términos)', controller_B), ('C (9 términos)', controller_C)):
    theta, u = simular(controller)
    plt.plot(t, theta, label=nombre)
plt.xlabel('Tiempo (s)')
plt.ylabel('Posición (rad)')
plt.title('Controladores base sin optimizar')
plt.grid(True)
plt.legend()
plt.show()

### **Fase 2:** Optimización mediante **PyGAD**

- 20 individuos
- 30 generaciones

### **Fase 3:** Selección del mejor individuo

- el mejor individuo del sistema de 5 términos;
- el mejor individuo del sistema de 7 términos;
- el mejor individuo del sistema de 9 términos.

### **Fase 4:** Evaluación del costo computacional

- **Tiempo total de optimización:** Tiempo requerido para completar todas las generaciones del algoritmo genético.
- **Tiempo promedio de inferencia:** Tiempo promedio requerido para generar una salida del controlador difuso.
- **Número de parámetros optimizados:** Cantidad total de genes presentes en el cromosoma.
- **El número y exactamente qué reglas quedaron activas:** Cantidad de reglas ﬁnalmente utilizadas por la mejor solución encontrada y además cuáles fueron seleccionadas.
- **Uso de memoria:** Uso aproximado de memoria asociado al controlador obtenido.

### **Fase 5:** Repetibilidad Experimental

Los algoritmos genéticos son métodos estocásticos. Por esta razón, todos los experimentos deben repetirse como mínimo 5 veces utilizando semillas aleatorias distintas.